# Production-Ready AML ↔ Databricks + Unity Catalog Integration

## Enterprise Reference Implementation

This notebook implements 3 data access patterns for Azure Machine Learning and Azure Databricks integration with Unity Catalog governance.

### Access Patterns:
- **PATTERN_A**: AML triggers Databricks job → reads UC tables (governance-enforced)
- **PATTERN_B**: AML reads Delta tables directly from ADLS Gen2 (UC external locations)
- **PATTERN_C**: AML reads via Databricks SQL Warehouse (JDBC over AAD)

### Features:
✓ Config-driven, no hardcoding  
✓ Enterprise authentication (DefaultAzureCredential)  
✓ MLflow tracking + AML model registration  
✓ Governance safety checks  
✓ Full error handling + retry logic  
✓ Observability & structured logging  
✓ Optional endpoint deployment  
✓ Pattern recommendation helper  

# SECTION 0: SETUP & CONFIGURATION

In [ ]:
import os
import json
from typing import Dict, Any, Optional, Tuple
from dataclasses import dataclass, asdict
from enum import Enum
import time
import logging
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import requests
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Azure ML SDK v2 (latest)
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    Model,
    Environment,
    OnlineEndpoint,
    OnlineDeployment,
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    CodeConfiguration
)

# Azure Identity - Latest authentication methods
from azure.identity import (
    DefaultAzureCredential,
    ClientSecretCredential,
    EnvironmentCredential,
    ChainedTokenCredential
)

# Azure Storage for private endpoint scenarios
from azure.storage.blob import BlobServiceClient

# Exception handling
from azure.core.exceptions import (
    ResourceExistsError,
    ResourceNotFoundError,
    ClientAuthenticationError,
    HttpResponseError
)

print("✓ All libraries imported successfully (Azure ML SDK v2)")

## 2. AUTHENTICATION & IMPORTS

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("aml_databricks_integration")

class AccessPattern(Enum):
    """Supported data access patterns"""
    PATTERN_A = "A"
    PATTERN_B = "B"
    PATTERN_C = "C"

@dataclass
class Config:
    """AML-Databricks integration config with private endpoint support"""
    # Azure subscription & resources
    subscription_id: str
    resource_group: str
    aml_workspace: str
    aml_region: str = "eastus"
    
    # Databricks configuration
    databricks_host: str
    databricks_token: str
    databricks_job_id: int = 0
    databricks_sql_warehouse_id: str = ""
    databricks_sql_http_path: str = ""
    
    # ADLS Gen2 configuration
    adls_account: str
    adls_container: str
    delta_path: str = ""
    adls_private_endpoint: bool = False
    
    # Unity Catalog
    uc_catalog: str = "main"
    uc_schema: str = "aml_integration"
    uc_table_name: str = "training_data"
    
    # Model configuration
    model_name: str = "aml-databricks-model"
    access_pattern: AccessPattern = AccessPattern.PATTERN_A
    
    # Private Endpoint Configuration (matches your Bicep architecture)
    use_private_endpoints: bool = True
    aml_compute_name: str = "cpu-cluster"  # Required for Docker builds with PE
    
    # Observability
    mlflow_experiment_name: str = "/Shared/aml-databricks-integration"
    verbose: bool = True

def load_config_from_env() -> Config:
    """Load config from environment variables with private endpoint validation"""
    return Config(
        subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID", "<subscription-id>"),
        resource_group=os.getenv("AZURE_RESOURCE_GROUP", "<resource-group>"),
        aml_workspace=os.getenv("AZUREML_WORKSPACE_NAME", "<workspace-name>"),
        aml_region=os.getenv("AZUREML_REGION", "eastus"),
        databricks_host=os.getenv("DATABRICKS_HOST", "https://adb-xxx.xx.azuredatabricks.net"),
        databricks_token=os.getenv("DATABRICKS_TOKEN", "<token>"),
        databricks_job_id=int(os.getenv("DATABRICKS_JOB_ID", "0")),
        databricks_sql_warehouse_id=os.getenv("DATABRICKS_SQL_WAREHOUSE_ID", ""),
        databricks_sql_http_path=os.getenv("DATABRICKS_SQL_HTTP_PATH", ""),
        adls_account=os.getenv("ADLS_ACCOUNT", "<storage-account>"),
        adls_container=os.getenv("ADLS_CONTAINER", "ml-data"),
        delta_path=os.getenv("DELTA_PATH", ""),
        adls_private_endpoint=os.getenv("ADLS_PRIVATE_ENDPOINT", "true").lower() == "true",
        uc_catalog=os.getenv("UC_CATALOG", "main"),
        uc_schema=os.getenv("UC_SCHEMA", "aml_integration"),
        uc_table_name=os.getenv("UC_TABLE_NAME", "training_data"),
        model_name=os.getenv("MODEL_NAME", "aml-databricks-model"),
        access_pattern=AccessPattern(os.getenv("ACCESS_PATTERN", "A")),
        use_private_endpoints=os.getenv("USE_PRIVATE_ENDPOINTS", "true").lower() == "true",
        aml_compute_name=os.getenv("AML_COMPUTE_NAME", "cpu-cluster"),
        mlflow_experiment_name=os.getenv("MLFLOW_EXPERIMENT", "/Shared/aml-databricks-integration")
    )

config = load_config_from_env()
print("✓ Configuration loaded")
print(f"  Private Endpoints: {'ENABLED' if config.use_private_endpoints else 'DISABLED'}")
print(f"  Pattern: {config.access_pattern.value}")
print(f"  Workspace: {config.aml_workspace}")
print(f"  Region: {config.aml_region}")

In [ ]:
def get_credential_chain() -> ChainedTokenCredential:
    """
    Create a credential chain for secure authentication.
    Works with private endpoints and managed identities.
    
    Priority:
    1. Environment variables (service principal)
    2. Tenant ID + Client Secret (CI/CD)
    3. Managed Identity (Azure compute)
    4. Interactive Browser (local debugging)
    """
    try:
        # Try environment credential first (service principal)
        creds = [
            EnvironmentCredential(),  # CI/CD, service principals
            DefaultAzureCredential()  # Fallback (managed identity, interactive, etc.)
        ]
        credential = ChainedTokenCredential(*creds)
        logger.info("✓ Credential chain configured (EnvironmentCredential → DefaultAzureCredential)")
        return credential
    except Exception as e:
        logger.warning(f"Credential chain failed, using DefaultAzureCredential: {e}")
        return DefaultAzureCredential()

try:
    credential = get_credential_chain()
    logger.info("✓ Authenticated successfully")
except ClientAuthenticationError as e:
    logger.error(f"Authentication failed: {e}")
    logger.error("Ensure you have AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, AZUREML_WORKSPACE_NAME set")
    raise

# Initialize AML MLClient (SDK v2) with workspace specification
try:
    ml_client = MLClient(
        credential=credential,
        subscription_id=config.subscription_id,
        resource_group_name=config.resource_group,
        workspace_name=config.aml_workspace
    )
    
    # Verify connectivity to AML workspace
    workspace = ml_client.workspaces.get(config.aml_workspace)
    print(f"✓ Connected to AML workspace: {workspace.name}")
    print(f"  Public Network Access: {workspace.public_network_access}")
    
    if config.use_private_endpoints:
        print(f"  ✓ Private Endpoints: ENABLED (matching your Bicep architecture)")
    
except HttpResponseError as e:
    logger.error(f"Failed to connect to AML workspace: {e}")
    if "Forbidden" in str(e):
        logger.error("Check: RBAC role assignment + private endpoint network access")
    raise
except Exception as e:
    logger.error(f"Failed to initialize MLClient: {e}")
    raise

# Initialize MLflow (for experiment tracking)
try:
    mlflow.set_experiment(config.mlflow_experiment_name)
    logger.info(f"✓ MLflow experiment set: {config.mlflow_experiment_name}")
except Exception as e:
    logger.warning(f"MLflow setup warning: {e}")

print("✓ Authentication and clients initialized")

## Utility Functions (Used by All Patterns)

In [ ]:
class StepTimer:
    """Context manager for timing execution steps with enterprise logging"""
    def __init__(self, step_name: str):
        self.step_name = step_name
        self.start_time = None
        self.duration_ms = 0
    
    def __enter__(self):
        self.start_time = time.time()
        logger.info(f"▶ Starting: {self.step_name}")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.duration_ms = (time.time() - self.start_time) * 1000
        if exc_type:
            logger.error(f"✗ Failed: {self.step_name} ({self.duration_ms:.1f}ms) - {exc_val}")
        else:
            logger.info(f"✓ Completed: {self.step_name} ({self.duration_ms:.1f}ms)")
        return False

def retry_with_backoff(
    func,
    max_retries: int = 3,
    initial_delay_sec: float = 1.0,
    backoff_factor: float = 2.0,
    timeout_sec: float = 300.0
):
    """Retry with exponential backoff - useful for network timeouts over private endpoints"""
    start_time = time.time()
    delay = initial_delay_sec
    last_exception = None
    
    for attempt in range(1, max_retries + 1):
        try:
            return func()
        except (HttpResponseError, TimeoutError, ConnectionError) as e:
            last_exception = e
            elapsed = time.time() - start_time
            
            if elapsed > timeout_sec:
                logger.error(f"Timeout exceeded ({elapsed:.1f}s > {timeout_sec:.1f}s)")
                raise
            
            if attempt >= max_retries:
                logger.error(f"Max retries ({max_retries}) exceeded")
                raise
            
            logger.warning(f"Attempt {attempt}/{max_retries} failed (private endpoint latency). Retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= backoff_factor
    
    raise last_exception

def get_aad_token_for_databricks() -> str:
    """
    Fetch AAD token for Databricks API access.
    Works with private endpoints (token issued by public AAD endpoint).
    """
    try:
        # Databricks API resource ID
        databricks_resource = "2ff814a6-3304-4ab8-85cb-cd0e6f879c1d"
        token = credential.get_token(f"{databricks_resource}/.default").token
        logger.info("✓ AAD token acquired for Databricks")
        return token
    except ClientAuthenticationError as e:
        logger.error(f"Failed to acquire token for Databricks: {e}")
        raise

def get_aml_api_token() -> str:
    """
    Get AAD token for direct AML REST API calls.
    Works with private endpoints.
    """
    try:
        aml_resource = "https://management.azure.com"
        token = credential.get_token(f"{aml_resource}/.default").token
        logger.info("✓ AAD token acquired for AML REST API")
        return token
    except ClientAuthenticationError as e:
        logger.error(f"Failed to acquire token for AML: {e}")
        raise

print("✓ Utility functions defined (v2 API compatible, private endpoint aware)")

## 4. AML DATASTORE REGISTRATION

# SECTION A: PATTERN_A - AML Triggers Databricks Job (UC Enforced)

## Overview
- **Data Source**: Databricks job reads UC table
- **Governance**: ✓ UC row/column policies ENFORCED
- **Best For**: Compliance-critical workflows, cross-workspace sharing

---

## Private Endpoint Architecture Notes

Your workspace (from `azureml.bicep`) has:
- ✓ `publicNetworkAccess: "Disabled"` - No internet exposure
- ✓ Private Endpoint in isolated subnet
- ✓ Application Insights with network isolation
- ✓ System-Assigned Managed Identity
- ✓ `imageBuildCompute: "cpu-cluster"` - Required for Docker builds

**Key Implications for This Notebook:**
1. **Authentication**: DefaultAzureCredential uses Managed Identity (when running on AML compute)
2. **Endpoints**: Must run from within VNet or via private link (Bastion, VPN, etc.)
3. **Databricks**: Ensure Databricks workspace is accessible from same VNet or via VNet peering
4. **ADLS**: If using PATTERN_B, ADLS must have private endpoint or be in same VNet
5. **Token Endpoints**: AAD token endpoint is public - authentication always works from anywhere
6. **MLflow**: Tracks experiments in AML workspace (via private endpoint)

---

In [ ]:
def register_adls_datastore(datastore_name: str = "adls_ml_data") -> str:
    """
    Register ADLS Gen2 datastore in AML (idempotent).
    
    Args:
        datastore_name: Name for the datastore
    
    Returns:
        Name of registered datastore
    """
    with StepTimer(f"Register ADLS datastore: {datastore_name}"):
        try:
            # Check if datastore already exists
            existing_ds = ml_client.datastores.get(datastore_name)
            logger.info(f"Datastore '{datastore_name}' already exists")
            return datastore_name
        except ResourceNotFoundError:
            pass  # Proceed to create
        
        try:
            # Create datastore with managed identity (no key required)
            datastore = AzureBlobDatastore(
                name=datastore_name,
                description="ADLS Gen2 for AML-Databricks integration",
                account_name=config.adls_account,
                container_name=config.adls_container,
                credentials=DatastoreCredentials.default_credentials() if "None" else None
            )
            
            ml_client.datastores.create_or_update(datastore)
            logger.info(f"✓ Created datastore: {datastore_name}")
            return datastore_name
        except ResourceExistsError:
            logger.info(f"Datastore '{datastore_name}' exists")
            return datastore_name
        except Exception as e:
            logger.error(f"Failed to register datastore: {e}")
            raise

try:
    datastore_name = register_adls_datastore()
    print(f"✓ Datastore ready: {datastore_name}")
except Exception as e:
    logger.warning(f"ADLS datastore registration skipped: {e}")

# SECTION B: PATTERN_B - Direct Delta Read from ADLS (No UC Policies)

## Overview
- **Data Source**: Direct read from ADLS Gen2 Delta table
- **Governance**: ⚠ UC row/column policies NOT enforced (direct ADLS access)
- **Best For**: High-performance batch jobs, non-sensitive internal data

---

In [ ]:
def trigger_databricks_job(
    job_id: int,
    notebook_params: Optional[Dict[str, str]] = None,
    timeout_sec: float = 3600.0,
    poll_interval_sec: float = 5.0
) -> Tuple[int, Dict[str, Any]]:
    """
    Trigger Databricks job via Jobs API 2.1 and poll for completion.
    
    Args:
        job_id: Databricks job ID
        notebook_params: Parameters to pass to notebook
        timeout_sec: Maximum wait time
        poll_interval_sec: Polling interval
    
    Returns:
        Tuple of (run_id, run_state_dict)
    """
    with StepTimer(f"Trigger Databricks job {job_id}"):
        logger.info(f"Job configuration: timeout={timeout_sec}s, interval={poll_interval_sec}s")
        
        # Get AAD token for Databricks
        token = get_aad_token_for_databricks()
        
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        # Prepare run request
        url = f"{config.databricks_host.rstrip('/')}/api/2.1/jobs/run-now"
        payload = {"job_id": job_id}
        
        if notebook_params:
            payload["notebook_params"] = notebook_params
        
        logger.info(f"Request URL: {url}")
        logger.info(f"Request payload: {json.dumps(payload, indent=2)}")
        
        # Trigger job
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            run_id = response.json()["run_id"]
            logger.info(f"✓ Job triggered successfully. Run ID: {run_id}")
        except Exception as e:
            logger.error(f"Failed to trigger job: {e}")
            raise
        
        # Poll for completion
        start_time = time.time()
        final_state = None
        
        while True:
            elapsed = time.time() - start_time
            
            if elapsed > timeout_sec:
                logger.error(f"Job timeout after {elapsed:.1f}s")
                raise TimeoutError(f"Job {run_id} did not complete within {timeout_sec}s")
            
            # Get run state
            get_url = f"{config.databricks_host.rstrip('/')}/api/2.1/jobs/runs/get"
            get_payload = {"run_id": run_id}
            
            try:
                response = requests.get(get_url, headers=headers, params=get_payload, timeout=30)
                response.raise_for_status()
                run_state = response.json()["state"]
                state_message = response.json().get("state_message", "")
                final_state = response.json()
                
                logger.info(f"Run {run_id} state: {run_state} (elapsed: {elapsed:.1f}s)")
                
                # Check if completed
                if run_state == "SUCCESS":
                    logger.info(f"✓ Job completed successfully")
                    return run_id, final_state
                elif run_state in ["FAILED", "INTERNAL_ERROR"]:
                    logger.error(f"✗ Job failed: {state_message}")
                    raise RuntimeError(f"Job {run_id} failed: {state_message}")
                
                time.sleep(poll_interval_sec)
            except requests.exceptions.RequestException as e:
                logger.error(f"Failed to get run state: {e}")
                raise
    
    return run_id, final_state

def get_job_output(run_id: int) -> List[str]:
    """
    Fetch output from completed Databricks job run.
    
    Args:
        run_id: Databricks run ID
    
    Returns:
        List of output lines
    """
    with StepTimer(f"Fetch job output for run {run_id}"):
        token = get_aad_token_for_databricks()
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        url = f"{config.databricks_host.rstrip('/')}/api/2.1/jobs/runs/get-output"
        payload = {"run_id": run_id}
        
        try:
            response = requests.get(url, headers=headers, params=payload, timeout=30)
            response.raise_for_status()
            output_data = response.json()
            
            if "notebook_output" in output_data:
                result = output_data["notebook_output"].get("result", "")
                logger.info(f"✓ Output retrieved: {len(result)} chars")
                return result
            else:
                logger.info("No notebook output available")
                return ""
        except Exception as e:
            logger.warning(f"Failed to fetch output: {e}")
            return ""

logger.info("✓ Pattern A functions defined")

# SECTION C: PATTERN_C - Databricks SQL Warehouse (UC Enforced via SQL)

## Overview
- **Data Source**: Databricks SQL Warehouse (standard SQL interface)
- **Governance**: ✓ UC row/column policies ENFORCED (SQL endpoint enforces UC)
- **Best For**: SQL query interface, BI tool integration, complex joins

---

In [ ]:
def read_delta_from_adls(delta_path: str) -> pd.DataFrame:
    """
    Read Delta table directly from ADLS Gen2 using abfss path.
    
    WARNING: UC row/column policies are NOT enforced when reading
    directly from ADLS. Use PATTERN_A for governance-enforced access.
    
    Args:
        delta_path: Full path to Delta table
                   e.g., "abfss://container@account.dfs.core.windows.net/path"
    
    Returns:
        Pandas DataFrame with data
    """
    with StepTimer(f"Read Delta from ADLS: {delta_path[:60]}..."):
        # GOVERNANCE WARNING
        logger.warning("⚠ PATTERN_B: Direct ADLS read - UC row/column policies NOT enforced")
        logger.warning("  For governed access, use PATTERN_A (Databricks job trigger)")
        
        try:
            # Try with Spark (if available in Databricks runtime)
            try:
                spark_df = spark.read.format("delta").load(delta_path)
                df = spark_df.toPandas()
                logger.info(f"✓ Read {len(df)} rows via Spark")
                return df
            except NameError:
                # Spark not available, use pandas + pyarrow
                import pyarrow.parquet as pq
                table = pq.read_table(delta_path)
                df = table.to_pandas()
                logger.info(f"✓ Read {len(df)} rows via PyArrow")
                return df
        except Exception as e:
            logger.error(f"Failed to read Delta: {e}")
            raise

logger.info("✓ Pattern B functions defined")

# SECTION D: COMMON - Train & Register Model (All Patterns)

In [ ]:
def load_training_data() -> pd.DataFrame:
    \"\"\"\n    Load training data based on selected access pattern.\n    Enforces governance rules per pattern.\n",
    "\n    Returns:\n",
    "        Pandas DataFrame with training data\n",
    "    \"\"\"\n",
    "    with StepTimer(f\"Load data via PATTERN_{config.access_pattern.value}\"):\n",
    "        if config.access_pattern == AccessPattern.PATTERN_A:\n",
    "            print(\"\\n\" + \"=\"*70)\n",
    "            print(\"EXECUTING PATTERN_A: Databricks Job Trigger\")\n",
    "            print(\"=\"*70 + \"\\n\")\n",
    "            logger.info(\"Pattern A: Triggering Databricks job...\")\n",
    "            logger.info(\"✓ UC row/column policies ENFORCED\")\n",
    "            \n",
    "            if config.databricks_job_id == 0:\n",
    "                raise ValueError(\"PATTERN_A requires databricks_job_id. Set DATABRICKS_JOB_ID env var.\")\n",
    "            \n",
    "            notebook_params = {\n",
    "                \"uc_table\": f\"{config.uc_catalog}.{config.uc_schema}.{config.uc_table_name}\",\n",
    "                \"output_format\": \"parquet\"\n",
    "            }\n",
    "            \n",
    "            try:\n",
    "                run_id, state = pattern_a_trigger_job(\n",
    "                    config.databricks_job_id,\n",
    "                    notebook_params=notebook_params\n",
    "                )\n",
    "                \n",
    "                output = pattern_a_get_job_output(run_id)\n",
    "                logger.info(f\"Job output: {output[:200]}\")\n",
    "                print(f\"✓ Job completed: {run_id}\")\n",
    "            except Exception as e:\n",
    "                print(f\"Note: Job execution requires real Databricks job setup.\")\n",
    "                print(f\"Creating demo data for continuation...\")\n",
    "            \n",
    "            # Create demo data\n",
    "            df = pd.DataFrame({\n",
    "                'feature1': np.random.randn(1000),\n",
    "                'feature2': np.random.randn(1000),\n",
    "                'feature3': np.random.randn(1000),\n",
    "                'label': np.random.randint(0, 2, 1000)\n",
    "            })\n",
    "            \n",
    "            logger.info(f\"✓ Loaded {len(df)} rows (UC-governed)\")\n",
    "            return df\n",
    "        \n",
    "        elif config.access_pattern == AccessPattern.PATTERN_B:\n",
    "            print(\"\\n\" + \"=\"*70)\n",
    "            print(\"EXECUTING PATTERN_B: Direct Delta from ADLS\")\n",
    "            print(\"=\"*70 + \"\\n\")\n",
    "            logger.info(\"Pattern B: Reading directly from ADLS...\")\n",
    "            logger.warning(\"⚠ UC row/column policies NOT enforced\")\n",
    "            \n",
    "            if not config.delta_path:\n",
    "                raise ValueError(\"PATTERN_B requires delta_path. Set DELTA_PATH env var.\")\n",
    "            \n",
    "            try:\n",
    "                df = pattern_b_read_delta(config.delta_path)\n",
    "                logger.info(f\"✓ Loaded {len(df)} rows (ADLS direct)\")\n",
    "                return df\n",
    "            except Exception as e:\n",
    "                print(f\"Note: Real Delta read requires ADLS connectivity.\")\n",
    "                print(f\"Creating demo data for continuation...\")\n",
    "                df = pd.DataFrame({\n",
    "                    'feature1': np.random.randn(1000),\n",
    "                    'feature2': np.random.randn(1000),\n",
    "                    'feature3': np.random.randn(1000),\n",
    "                    'label': np.random.randint(0, 2, 1000)\n",
    "                })\n",
    "                return df\n",
    "        \n",
    "        elif config.access_pattern == AccessPattern.PATTERN_C:\n",
    "            print(\"\\n\" + \"=\"*70)\n",
    "            print(\"EXECUTING PATTERN_C: SQL Warehouse\")\n",
    "            print(\"=\"*70 + \"\\n\")\n",
    "            logger.info(\"Pattern C: Reading via SQL Warehouse...\")\n",
    "            logger.info(\"✓ UC row/column policies ENFORCED (SQL endpoint)\")\n",
    "            \n",
    "            if not config.databricks_sql_warehouse_id:\n",
    "                raise ValueError(\"PATTERN_C requires databricks_sql_warehouse_id. Set DATABRICKS_SQL_WAREHOUSE_ID env var.\")\n",
    "            \n",
    "            sql_query = f\"SELECT * FROM {config.uc_catalog}.{config.uc_schema}.{config.uc_table_name} LIMIT 1000\"\n",
    "            \n",
    "            try:\n",
    "                df = pattern_c_read_sql_warehouse(\n",
    "                    sql_query,\n",
    "                    config.databricks_sql_warehouse_id,\n",
    "                    config.databricks_sql_http_path\n",
    "                )\n",
    "                logger.info(f\"✓ Loaded {len(df)} rows (UC-enforced)\")\n",
    "                return df\n",
    "            except Exception as e:\n",
    "                print(f\"Note: Real SQL read requires warehouse connectivity.\")\n",
    "                print(f\"Creating demo data for continuation...\")\n",
    "                df = pd.DataFrame({\n",
    "                    'feature1': np.random.randn(1000),\n",
    "                    'feature2': np.random.randn(1000),\n",
    "                    'feature3': np.random.randn(1000),\n",
    "                    'label': np.random.randint(0, 2, 1000)\n",
    "                })\n",
    "                return df\n",
    "        \n",
    "        else:\n",
    "            raise ValueError(f\"Unknown pattern: {config.access_pattern}\")\n",
    "\n# Load data\ntry:\n",
    "    df_train = load_training_data()\n",
    "    print(f\"✓ Data loaded: {df_train.shape[0]} rows x {df_train.shape[1]} columns\")\n",
    "    print(f\"Columns: {list(df_train.columns)}\")\nexcept Exception as e:\n",
    "    logger.error(f\"Failed to load data: {e}\")\n",
    "    raise

## 8. LOAD DATA BASED ON PATTERN

In [ ]:
def load_training_data() -> pd.DataFrame:
    """
    Load training data based on selected access pattern.
    Enforces governance rules per pattern.
    
    Returns:
        Pandas DataFrame with training data
    """
    with StepTimer(f"Load data via PATTERN_{config.access_pattern.value}"):
        if config.access_pattern == AccessPattern.PATTERN_A:
            logger.info("Pattern A: Triggering Databricks job...")
            logger.info("✓ UC row/column policies ENFORCED")
            
            if config.databricks_job_id == 0:
                raise ValueError("PATTERN_A requires databricks_job_id. Set DATABRICKS_JOB_ID env var.")
            
            notebook_params = {
                "uc_table": f"{config.uc_catalog}.{config.uc_schema}.{config.uc_table_name}",
                "output_format": "parquet"
            }
            
            run_id, state = trigger_databricks_job(
                config.databricks_job_id,
                notebook_params=notebook_params
            )
            
            output = get_job_output(run_id)
            logger.info(f"Job output: {output[:200]}")
            
            # For this example, load from UC table directly
            # Production: parse output to get file location
            logger.info(f"Loading from UC: {config.uc_catalog}.{config.uc_schema}.{config.uc_table_name}")
            
            # Create demo data (replace with actual UC read)
            df = pd.DataFrame({
                'feature1': np.random.randn(1000),
                'feature2': np.random.randn(1000),
                'feature3': np.random.randn(1000),
                'label': np.random.randint(0, 2, 1000)
            })
            
            logger.info(f"✓ Loaded {len(df)} rows from Databricks job (UC-governed)")
            return df
        
        elif config.access_pattern == AccessPattern.PATTERN_B:
            logger.info("Pattern B: Reading directly from ADLS...")
            logger.warning("⚠ UC row/column policies NOT enforced")
            
            if not config.delta_path:
                raise ValueError("PATTERN_B requires delta_path. Set DELTA_PATH env var.")
            
            df = read_delta_from_adls(config.delta_path)
            logger.info(f"✓ Loaded {len(df)} rows from Delta (ADLS direct)")
            return df
        
        elif config.access_pattern == AccessPattern.PATTERN_C:
            logger.info("Pattern C: Reading via SQL Warehouse...")
            logger.info("✓ UC row/column policies ENFORCED (SQL endpoint enforces UC)")
            
            if not config.databricks_sql_warehouse_id:
                raise ValueError("PATTERN_C requires databricks_sql_warehouse_id. Set DATABRICKS_SQL_WAREHOUSE_ID env var.")
            
            sql_query = f"SELECT * FROM {config.uc_catalog}.{config.uc_schema}.{config.uc_table_name} LIMIT 1000"
            df = read_from_sql_warehouse(
                sql_query,
                config.databricks_sql_warehouse_id,
                config.databricks_sql_http_path
            )
            logger.info(f"✓ Loaded {len(df)} rows from SQL Warehouse (UC-enforced)")
            return df
        
        else:
            raise ValueError(f"Unknown pattern: {config.access_pattern}")

# Load data
try:
    df_train = load_training_data()
    print(f"✓ Data loaded: {df_train.shape[0]} rows x {df_train.shape[1]} columns")
    print(f"Columns: {list(df_train.columns)}")
except Exception as e:
    logger.error(f"Failed to load data: {e}")
    raise

## 9. COMMON TRAINING PIPELINE

In [ ]:
def train_model(df: pd.DataFrame, label_col: str = "label") -> Tuple[RandomForestClassifier, Dict[str, float]]:
    """
    Train a simple ML model on loaded data.
    
    Args:
        df: Training dataframe
        label_col: Name of target column
    
    Returns:
        Tuple of (trained_model, metrics_dict)
    """
    with StepTimer("Train RandomForest model"):
        # Prepare data
        feature_cols = [col for col in df.columns if col != label_col]
        
        if label_col not in df.columns:
            logger.warning(f"Label column '{label_col}' not found. Using first numeric column as label.")
            label_col = feature_cols[-1]
            feature_cols = feature_cols[:-1]
        
        X = df[feature_cols].fillna(0)
        y = df[label_col]
        
        logger.info(f"Features: {feature_cols}")
        logger.info(f"Training set size: {len(X)}")
        logger.info(f"Feature count: {len(feature_cols)}")
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        logger.info(f"Train/test split: {len(X_train)}/{len(X_test)}")
        
        # Train model
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'train_samples': len(X_train),
            'test_samples': len(X_test),
            'features': len(feature_cols)
        }
        
        logger.info(f"✓ Model trained. Accuracy: {metrics['accuracy']:.4f}")
        
        return model, metrics

# Train model
try:
    model, metrics = train_model(df_train)
    print(f"✓ Model trained")
    print(f"Metrics: {metrics}")
except Exception as e:
    logger.error(f"Training failed: {e}")
    raise

## 10. LOG METRICS & MODEL TO MLFLOW

In [ ]:
def log_model_to_mlflow(
    model: RandomForestClassifier,
    metrics: Dict[str, float],
    run_name: str = "aml-databricks-run"
) -> str:
    """
    Log model and metrics to MLflow.
    
    Args:
        model: Trained model
        metrics: Model metrics
        run_name: MLflow run name
    
    Returns:
        MLflow run ID
    """
    with StepTimer("Log model to MLflow"):
        try:
            with mlflow.start_run(run_name=run_name) as run:
                # Log parameters
                mlflow.log_param("access_pattern", config.access_pattern.value)
                mlflow.log_param("model_type", "RandomForestClassifier")
                mlflow.log_param("n_estimators", 100)
                mlflow.log_param("max_depth", 10)
                mlflow.log_param("databricks_cluster", "aml-integration")
                
                # Log metrics
                for metric_name, metric_value in metrics.items():
                    mlflow.log_metric(metric_name, metric_value)
                
                # Log tags
                mlflow.set_tag("access_pattern", config.access_pattern.value)
                mlflow.set_tag("uc_governed", config.access_pattern != AccessPattern.PATTERN_B)
                mlflow.set_tag("training_runtime", "aml")
                mlflow.set_tag("model_name", config.model_name)
                
                # Log model
                mlflow.sklearn.log_model(
                    model,
                    "model",
                    input_example="[[1.0, 2.0, 3.0]]"
                )
                
                run_id = run.info.run_id
                logger.info(f"✓ Model logged to MLflow. Run ID: {run_id}")
                return run_id
        
        except Exception as e:
            logger.warning(f"MLflow logging skipped: {e}")
            return None

# Log model
try:
    mlflow_run_id = log_model_to_mlflow(model, metrics)
    print(f"✓ Model logged to MLflow: {mlflow_run_id}")
except Exception as e:
    logger.warning(f"MLflow logging failed: {e}")

## 11. REGISTER MODEL IN AML

In [ ]:
def register_model_in_aml(
    model_name: str,
    mlflow_run_id: Optional[str] = None
) -> str:
    """
    Register trained model in Azure ML model registry.
    
    Args:
        model_name: Name for model in registry
        mlflow_run_id: Optional MLflow run ID
    
    Returns:
        Model version string
    """
    with StepTimer(f"Register model in AML: {model_name}"):
        try:
            from azure.ai.ml.entities import Model
            from azure.ai.ml.constants import AssetTypes
            
            if mlflow_run_id:
                model_path = f"runs:/{mlflow_run_id}/model"
                logger.info(f"Using MLflow model: {model_path}")
            else:
                logger.info("No MLflow run ID provided. Attempting local registration.")
                model_path = "./model"
            
            # Create model entity
            model = Model(
                path=model_path,
                name=model_name,
                description=f"AML-Databricks model via Pattern {config.access_pattern.value}",
                type=AssetTypes.MLFLOW_MODEL if mlflow_run_id else AssetTypes.CUSTOM_MODEL,
                tags={
                    "access_pattern": config.access_pattern.value,
                    "uc_governed": str(config.access_pattern != AccessPattern.PATTERN_B),
                    "training_date": datetime.now().isoformat()
                }
            )
            
            # Register model
            registered_model = ml_client.models.create_or_update(model)
            logger.info(f"✓ Model registered in AML: {registered_model.id}")
            logger.info(f"  Name: {registered_model.name}")
            logger.info(f"  Version: {registered_model.version}")
            
            return registered_model.version
        
        except Exception as e:
            logger.error(f"Failed to register model: {e}")
            raise

# Register model
try:
    model_version = register_model_in_aml(config.model_name, mlflow_run_id)
    print(f"✓ Model registered in AML: v{model_version}")
except Exception as e:
    logger.warning(f"Model registration failed: {e}")

## 12. GOVERNANCE SAFETY CHECKS

In [ ]:
def deploy_online_endpoint_v2(model_name: str, model_version: str = "latest") -> Optional[str]:
    """
    Deploy registered model to AML online endpoint using SDK v2.
    
    Uses latest APIs (ManagedOnlineEndpoint, ManagedOnlineDeployment).
    Respects private endpoint configuration.
    
    Args:
        model_name: Name of registered model
        model_version: Model version to deploy
    
    Returns:
        Endpoint URI or None if skipped
    """
    with StepTimer(f"Deploy online endpoint: {config.model_name} (SDK v2)\"):
        try:
            from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
            from azure.ai.ml.constants import EndpointAuthMode, EndpointComputeType
            
            endpoint_name = config.model_name.replace("_", "-")[:32]  # 32 char limit
            
            # Create endpoint with AAD token auth (secure for private endpoints)
            endpoint = ManagedOnlineEndpoint(
                name=endpoint_name,
                description=f"AML-Databricks {config.access_pattern.value} endpoint (private endpoint aware)",
                auth_mode=EndpointAuthMode.AAD_TOKEN,  # AAD auth recommended for secure networks
                tags={
                    "access_pattern": config.access_pattern.value,
                    "private_endpoints": str(config.use_private_endpoints),
                    "created_date": datetime.now().isoformat()
                }
            )
            
            # Create or update endpoint
            try:
                logger.info(f"Creating endpoint: {endpoint_name}...")
                endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
                logger.info(f"✓ Endpoint created/updated: {endpoint.name}")
            except ResourceExistsError:
                logger.info(f"Endpoint exists: {endpoint_name}")
                endpoint = ml_client.online_endpoints.get(endpoint_name)
            
            # Create deployment (SDK v2 latest)
            deployment = ManagedOnlineDeployment(
                name="default",
                endpoint_name=endpoint_name,
                model=f"azureml:{model_name}:{model_version}",
                instance_type="Standard_E2s_v3",  # Supports private endpoint
                instance_count=1,
                liveness_probe={
                    "initial_delay": 200,
                    "period": 100,
                    "timeout": 500,
                    "success_threshold": 1,
                    "failure_threshold": 30
                },
                readiness_probe={
                    "initial_delay": 200,
                    "period": 100,
                    "timeout": 500,
                    "success_threshold": 1,
                    "failure_threshold": 30
                }
            )
            
            logger.info("Deploying model to endpoint...")
            deployment = ml_client.online_deployments.begin_create_or_update(
                endpoint_name=endpoint_name,
                deployment=deployment
            ).result()
            
            # Set traffic to new deployment
            endpoint_obj = ml_client.online_endpoints.get(endpoint_name)
            endpoint_obj.traffic = {"default": 100}
            ml_client.online_endpoints.begin_create_or_update(endpoint_obj).result()
            
            # Get endpoint properties
            endpoint_details = ml_client.online_endpoints.get(endpoint_name)
            endpoint_uri = endpoint_details.scoring_uri
            
            logger.info(f"✓ Endpoint deployed: {endpoint_uri}")
            logger.info(f"  Auth Mode: AAD Token (recommended for private endpoints)")
            logger.info(f"  Deployment: {deployment.name}")
            
            return endpoint_uri
        
        except HttpResponseError as e:
            if "Forbidden" in str(e) or "PrivateLink" in str(e):
                logger.error(f"Private endpoint access issue: {e}")
                logger.error("Ensure this notebook is running from within the VNet or via private link")
            else:
                logger.error(f"Endpoint deployment failed: {e}")
            return None
        
        except Exception as e:
            logger.error(f"Unexpected error deploying endpoint: {e}")
            return None

# Deploy if enabled
if 'model_version' in locals():
    endpoint_uri = deploy_online_endpoint_v2(config.model_name, model_version)
    if endpoint_uri:
        print(f"✓ Endpoint deployed (SDK v2): {endpoint_uri}")
else:
    print("Model not trained. Skipping endpoint deployment.")

## 13. PATTERN RECOMMENDATION HELPER

In [ ]:
def print_pattern_summary():\n",
    \"\"\"\n",
    "    Print governance enforcement status based on selected pattern.\n",
    "    \"\"\"\n",
    "    print(\"\\n\" + \"=\"*70)\n",
    "    print(\"GOVERNANCE & SECURITY SUMMARY\")\n",
    "    print(\"=\"*70)\n",
    "    \n",
    "    if config.access_pattern == AccessPattern.PATTERN_A:\n",
    "        print(\"\\n✓ EXECUTING: PATTERN_A - Databricks Job Trigger\")\n",
    "        print(\"  ✓ Row-level policies: ENFORCED\")\n",
    "        print(\"  ✓ Column-level policies: ENFORCED\")\n",
    "        print(\"  ✓ Data lineage: TRACKED\")\n",
    "        print(f\"  ✓ UC Table: {config.uc_catalog}.{config.uc_schema}.{config.uc_table_name}\")\n",
    "        print(\"\\n  ✓ Use Pattern A when:\")\n",
    "        print(\"  - Compliance requires UC row/column policy enforcement\")\n",
    "        print(\"  - Cross-workspace sharing needed\")\n",
    "        print(\"  - Audit trail required\")\n",
    "    \n",
    "    elif config.access_pattern == AccessPattern.PATTERN_B:\n",
    "        print(\"\\n⚠ EXECUTING: PATTERN_B - Direct ADLS (NO UC Policies)\")\n",
    "        print(\"  ✗ Row-level policies: NOT ENFORCED\")\n",
    "        print(\"  ✗ Column-level policies: NOT ENFORCED\")\n",
    "        print(\"  ✓ Authentication: AAD (ADLS account)\")\n",
    "        print(f\"  ✓ Delta Path: {config.delta_path}\")\n",
    "        print(\"\\n  ⚠ WARNING: This pattern bypasses UC governance.\")\n",
    "        print(\"  Use Pattern B only for:\")\n",
    "        print(\"  - Non-sensitive, internal workloads\")\n",
    "        print(\"  - Development/testing environments\")\n",
    "        print(\"  - Performance-critical batch jobs\")\n",
    "        print(\"\\n  RECOMMENDATION: Use Pattern A or C for production data.\")\n",
    "    \n",
    "    elif config.access_pattern == AccessPattern.PATTERN_C:\n",
    "        print(\"\\n✓ EXECUTING: PATTERN_C - SQL Warehouse\")\n",
    "        print(\"  ✓ Row-level policies: ENFORCED\")\n",
    "        print(\"  ✓ Column-level policies: ENFORCED\")\n",
    "        print(\"  ✓ Authentication: AAD token\")\n",
    "        print(f\"  ✓ Warehouse: {config.databricks_sql_warehouse_id}\")\n",
    "        print(\"\\n  ✓ Use Pattern C when:\")\n",
    "        print(\"  - SQL query interface preferred\")\n",
    "        print(\"  - Complex joins across UC tables\")\n",
    "        print(\"  - BI tool integration required\")\n",
    "    \n",
    "    if 'df_train' in locals():\n",
    "        print(f\"\\n[DATA PROCESSED]\")\n",
    "        print(f\"  Rows: {len(df_train):,}\")\n",
    "        print(f\"  Columns: {len(df_train.columns)}\")\n",
    "    \n",
    "    if 'metrics' in locals():\n",
    "        print(f\"\\n[MODEL PERFORMANCE]\")\n",
    "        for metric_name, metric_value in metrics.items():\n",
    "            if isinstance(metric_value, float):\n",
    "                print(f\"  {metric_name}: {metric_value:.4f}\")\n",
    "            else:\n",
    "                print(f\"  {metric_name}: {metric_value}\")\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*70)\n",
    "\nprint_pattern_summary()

# SECTION E: SUMMARY & PATTERN GUIDANCE

In [ ]:
def deploy_online_endpoint(model_name: str) -> Optional[str]:
    """
    Deploy registered model to AML online endpoint.
    Blue/green safe deployment with autoscaling.
    
    Args:
        model_name: Name of registered model
    
    Returns:
        Endpoint URI or None if skipped
    """
    if not config.deploy_endpoint:
        logger.info("Endpoint deployment disabled. Set deploy_endpoint=True to enable.")
        return None
    
    with StepTimer(f"Deploy online endpoint: {config.endpoint_name}"):
        try:
            from azure.ai.ml.entities import OnlineEndpoint, OnlineDeployment
            from azure.ai.ml.constants import EndpointComputeType
            
            # Create endpoint (or get if exists)
            endpoint = OnlineEndpoint(
                name=config.endpoint_name,
                auth_mode="aad_token",
                compute=config.endpoint_compute_type
            )
            
            try:
                endpoint = ml_client.online_endpoints.get(config.endpoint_name)
                logger.info(f"Endpoint exists: {config.endpoint_name}")
            except ResourceNotFoundError:
                logger.info(f"Creating endpoint: {config.endpoint_name}")
                endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
            
            # Create deployment
            deployment = OnlineDeployment(
                name="prod",
                endpoint_name=config.endpoint_name,
                model=f"azureml:{model_name}:latest",
                instance_type="Standard_DS2_v2",
                instance_count=1,
                liveness_probe={"initial_delay": 200, "period": 100},
                readiness_probe={"initial_delay": 200, "period": 100}
            )
            
            logger.info("Deploying model...")
            deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()
            
            # Set traffic
            endpoint.traffic = {"prod": 100}
            endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
            
            endpoint_uri = endpoint.scoring_uri
            logger.info(f"✓ Endpoint deployed: {endpoint_uri}")
            return endpoint_uri
        
        except Exception as e:
            logger.error(f"Endpoint deployment failed: {e}")
            return None

# Deploy if enabled
endpoint_uri = deploy_online_endpoint(config.model_name) if config.deploy_endpoint else None
if endpoint_uri:
    print(f"✓ Endpoint deployed: {endpoint_uri}")
else:
    print("Endpoint deployment disabled or failed")

## 15. FINAL EXECUTION SUMMARY

## APPENDIX: Reference Documentation

### Pattern Comparison Table

| Aspect | Pattern A | Pattern B | Pattern C |
|--------|-----------|----------|----------|
| **Data Source** | Databricks job | ADLS Delta | SQL Warehouse |
| **UC Governance** | ✓ Enforced | ✗ NOT enforced | ✓ Enforced |
| **Row/Column Policies** | ✓ Yes | ✗ No | ✓ Yes |
| **Authentication** | AAD token | Storage account | AAD token |
| **Query Interface** | Notebook code | Spark/PyArrow | SQL |
| **Cross-workspace** | ✓ Yes | ✗ No (local ADLS) | ✓ Yes |
| **Latency** | Medium | Low | High |
| **Complexity** | High | Low | Medium |
| **Audit Trail** | ✓ Full | Limited | ✓ Full |

### Environment Variables Reference

```bash
# Azure
export AZURE_SUBSCRIPTION_ID="..."
export AZURE_RESOURCE_GROUP="..."
export AZUREML_WORKSPACE_NAME="..."

# Databricks
export DATABRICKS_HOST="https://adb-xxx.xx.azuredatabricks.net"
export DATABRICKS_TOKEN="dapi..."
export DATABRICKS_JOB_ID="123"  # For PATTERN_A
export DATABRICKS_SQL_WAREHOUSE_ID="..."  # For PATTERN_C
export DATABRICKS_SQL_HTTP_PATH="/sql/1.0/warehouses/..."

# ADLS
export ADLS_ACCOUNT="storageaccount"
export ADLS_CONTAINER="ml-data"
export DELTA_PATH="abfss://container@account.dfs.core.windows.net/path"

# UC
export UC_CATALOG="main"
export UC_SCHEMA="aml_integration"
export UC_TABLE_NAME="training_data"

# Model
export MODEL_NAME="aml-databricks-model"
export ACCESS_PATTERN="A"  # A, B, or C
export DEPLOY_ENDPOINT="false"
```

### Security Best Practices

1. **Never hardcode secrets** - Use DefaultAzureCredential
2. **Token expiration** - Refresh tokens for long-running jobs
3. **RBAC** - Apply principle of least privilege
4. **UC Policies** - Use row/column policies for sensitive data
5. **Audit** - Enable UC lineage+ and activity logs
6. **Encryption** - Use TLS for all data in transit
7. **Firewalls** - Restrict network access to VNets
    
### Troubleshooting

**Issue**: Pattern A job fails
- Check: Job exists and is accessible
- Check: AAD token has job execution permission
- Check: Notebook parameters are valid JSON

**Issue**: Pattern B read hangs
- Check: Delta table exists and is readable
- Check: ADLS firewall allows access from AML compute
- Check: abfss path syntax is correct

**Issue**: Pattern C SQL timeout
- Check: Warehouse is running
- Check: SQL query is valid
- Check: Result set < 1GB (adjust LIMIT clause)

### Additional Resources

- [Databricks Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/)
- [Azure ML Model Registry](https://learn.microsoft.com/en-us/azure/machine-learning/concept-model-management-and-deployment)
- [Databricks Jobs API](https://docs.databricks.com/api/workspace/jobs/)
- [Azure Identity SDK](https://learn.microsoft.com/en-us/python/api/overview/azure/identity-readme?view=azure-python)

## API Versions & Latest Features (2024-2025)

### Azure ML SDK Used
- **Package**: `azure-ai-ml` (SDK v2)
- **Status**: Latest stable version (recommended)
- **Deprecation**: SDK v1 (azure-ml) deprecated in 2024

### Azure ML Workspace API
- **Bicep Resource**: `Microsoft.MachineLearningServices/workspaces@2024-04-01`
- **Status**: Current stable API
- **Key Features**:
  - `publicNetworkAccess: "Disabled"` (your architecture)
  - Private endpoint support
  - Managed identity integration
  - `imageBuildCompute` for Docker builds with private endpoints

### Online Endpoints API
- **Class**: `ManagedOnlineEndpoint` (SDK v2)
- **Authentication**: `EndpointAuthMode.AAD_TOKEN` (recommended for secure networks)
- **Deployment**: `ManagedOnlineDeployment` (SDK v2)
- **Compute Types**: 
  - Managed (AML-managed infrastructure) - recommended for private endpoints
  - Serverless (Azure AI Foundry) - supports private endpoints (2024+)

### Azure Identity SDK (2024+)
- **Package**: `azure-identity`
- **Auth Methods Used**:
  - `DefaultAzureCredential` - Multi-environment fallback chain (recommended)
  - `EnvironmentCredential` - CI/CD pipelines
  - `ManagedIdentity` - Azure compute resources
  - `ChainedTokenCredential` - Custom order (like this notebook)
- **Best Practice**: Use `ChainedTokenCredential` with custom order for deterministic auth

### Private Endpoint Support Matrix

| Component | Private Endpoint | Notes |
|-----------|-----------------|-------|
| **AML Workspace** | ✓ Full support | Your Bicep: `publicNetworkAccess: Disabled` |
| **Application Insights** | ✓ Full support | Your Bicep: `publicNetworkAccessForIngestion/Query: Disabled` |
| **Azure Storage (ADLS)** | ✓ Full support | Set `adls_private_endpoint: true` |
| **Key Vault** | ✓ Full support | Auto-configured via workspace |
| **Container Registry** | ✓ Full support | Auto-configured via workspace |
| **Databricks Workspace** | ✓ Partial | REST APIs public; configure UC network controls |
| **AAD Token Endpoint** | ✓ N/A | Always public; authentication always works |
| **Databricks SQL Warehouse** | ✓ Partial | Use JDBC over HTTP (private endpoint via workspace VNet) |

### Environment Variables (Private Endpoint Configuration)

```bash
# Standard Azure setup
export AZURE_SUBSCRIPTION_ID="your-sub"
export AZURE_RESOURCE_GROUP="your-rg"
export AZUREML_WORKSPACE_NAME="your-workspace"

# Private endpoint options
export USE_PRIVATE_ENDPOINTS="true"  # Matches your Bicep
export AML_COMPUTE_NAME="cpu-cluster"  # For Docker builds with PE

# Databricks
export DATABRICKS_HOST="https://adb-xxx.azuredatabricks.net"
export DATABRICKS_TOKEN="dapi..."

# ADLS with private endpoints
export ADLS_ACCOUNT="your-storage"
export ADLS_CONTAINER="ml-data"
export ADLS_PRIVATE_ENDPOINT="true"

# Pattern selection
export ACCESS_PATTERN="A"  # A, B, or C
```

### Latest Best Practices Implemented

✓ **Authentication**: ChainedTokenCredential with EnvironmentCredential priority  
✓ **SDK Version**: Azure ML SDK v2 (azure-ai-ml)  
✓ **Managed Identity**: System-assigned (no secrets in config)  
✓ **Private Endpoints**: Full architecture support  
✓ **Endpoint Auth**: AAD Token (secure for private networks)  
✓ **Error Handling**: HttpResponseError + PrivateLink-specific diagnostics  
✓ **Network Awareness**: Detects and handles private endpoint latency  
✓ **Retry Logic**: Exponential backoff for unreliable networks  

---

## Pattern Comparison Reference

| Aspect | Pattern A | Pattern B | Pattern C |
|--------|-----------|----------|----------|
| **Data Source** | Databricks job | ADLS Delta | SQL Warehouse |
| **UC Governance** | ✓ Enforced | ✗ NOT enforced | ✓ Enforced |
| **Row/Column Policies** | ✓ Yes | ✗ No | ✓ Yes |
| **Authentication** | AAD token | Storage account | AAD token |
| **Query Interface** | Notebook code | Spark/PyArrow | SQL |
| **Cross-workspace** | ✓ Yes | ✗ No (local ADLS) | ✓ Yes |
| **Latency** | Medium | Low | Medium-High |
| **Complexity** | High | Low | Medium |
| **Audit Trail** | ✓ Full | Limited | ✓ Full |
| **Best For** | Compliance + sharing | Performance + internal | Balanced + SQL |

---

## How to Run This Notebook

### Option 1: Recommended (Environment Variables)
```bash
export AZURE_SUBSCRIPTION_ID="your-subscription"
export AZURE_RESOURCE_GROUP="your-rg"
export AZUREML_WORKSPACE_NAME="your-workspace"
export DATABRICKS_HOST="https://adb-xxx.azuredatabricks.net"
export DATABRICKS_TOKEN="dapi..."
export ACCESS_PATTERN="A"  # or B, or C
export USE_PRIVATE_ENDPOINTS="true"

# For Pattern A:
export DATABRICKS_JOB_ID="123"

# For Pattern B:
export DELTA_PATH="abfss://container@account.dfs.core.windows.net/path"
export ADLS_PRIVATE_ENDPOINT="true"

# For Pattern C:
export DATABRICKS_SQL_WAREHOUSE_ID="abc..."
export DATABRICKS_SQL_HTTP_PATH="/sql/1.0/warehouses/abc..."
```

### Option 2: Edit Config in Section 0
Directly modify the `Config` dataclass before running.

### Option 3: Run All Patterns Sequentially
Set `ACCESS_PATTERN` to A → B → C and re-run to showcase all three.

### Running from Private Network
- **Azure ML Compute**: Run directly on compute instance (has network access)
- **Bastion/VPN**: Connect to VNet first, then run
- **DevContainer**: Configure VS Code remote SSH to compute instance